# Phase 2: Data Cleaning - FINAL VERSION

**Creates:**
- `cleaned_games` table (game-level) → For Phase 3 (EDA) & Phase 4 (ML Part 1)
- `cleaned_sale_events` table (event-level) → For Phase 5 (ML Part 2)

**100% Compatible with all notebooks**

In [48]:
import sys
import sqlite3
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

# Bootstrap: make `src` importable
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.notebook_setup import setup_notebook
conn, paths = setup_notebook()

Connected to: c:\Users\Sam\Documents\College\2026 Spring Term\DMW\Y2T2-Final-Project\data\steam.db
Tables (15): ['app_list', 'cleaned_discount_panel', 'cleaned_games', 'cleaned_sale_events', 'game_categories', 'game_genres', 'games', 'itad_mapping', 'player_counts', 'price_history', 'review_timestamps', 'reviews_summary', 'steamcharts_history', 'steamspy', 'steamspy_tags']


In [49]:
# Configuration — DB connection set up by setup_notebook above.
# Prices in this DB are already in PHP centavos (Storefront returns PHP because
# the collector queries with cc=ph). Just /100 to get pesos. No USD conversion.

---
# PART 1: GAME-LEVEL DATASET
---

In [50]:
print("Loading ALL data...")
df_games = pd.read_sql_query("SELECT * FROM games", conn)
df_steamspy = pd.read_sql_query("SELECT * FROM steamspy", conn)
df_reviews = pd.read_sql_query("SELECT * FROM reviews_summary", conn)
df_genres = pd.read_sql_query("SELECT * FROM game_genres", conn)
df_categories = pd.read_sql_query("SELECT * FROM game_categories", conn)
df_tags = pd.read_sql_query("SELECT * FROM steamspy_tags", conn)

genres_agg = df_genres.groupby('appid')['genre'].apply(list).reset_index()
genres_agg.columns = ['appid', 'genres']
categories_agg = df_categories.groupby('appid')['category'].apply(list).reset_index()
categories_agg.columns = ['appid', 'categories']
tags_agg = df_tags.groupby('appid')['tag'].apply(list).reset_index()
tags_agg.columns = ['appid', 'tags']

print(f"  Games: {len(df_games):,}")
print(f"  SteamSpy: {len(df_steamspy):,}")
print(f"  Reviews: {len(df_reviews):,}")

Loading ALL data...
  Games: 4,946
  SteamSpy: 4,946
  Reviews: 4,946


In [51]:
# Merge
df_games = df_games.drop_duplicates(subset='appid', keep='first')
df = df_games.merge(df_steamspy[['appid', 'owners_min', 'owners_max', 'average_forever', 'average_2weeks']], on='appid', how='left')
df = df.merge(df_reviews[['appid', 'total_reviews', 'total_positive', 'total_negative']], on='appid', how='left')
df = df.merge(genres_agg, on='appid', how='left')
df = df.merge(categories_agg, on='appid', how='left')
df = df.merge(tags_agg, on='appid', how='left')
print(f"✓ Merged: {len(df):,} games")

✓ Merged: 4,946 games


In [52]:
# Fill missing values
df['primary_genre_temp'] = df['genres'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else 'Unknown')
df['has_metacritic'] = df['metacritic_score'].notna().astype(int)
df['metacritic_score'] = df['metacritic_score'].fillna(df.groupby('primary_genre_temp')['metacritic_score'].transform('median')).fillna(df['metacritic_score'].median())
df['achievements_total'] = df['achievements_total'].fillna(0).astype(int)
df['total_reviews'] = df['total_reviews'].fillna(0).astype(int)
df['total_positive'] = df['total_positive'].fillna(0).astype(int)
df['total_negative'] = df['total_negative'].fillna(0).astype(int)
print("✓ Missing values filled")

✓ Missing values filled


In [53]:
# Age features (TIMEZONE-AWARE)
def parse_release_date(date_str):
    if pd.isna(date_str) or date_str == '':
        return pd.NaT
    for fmt in ['%b %d, %Y', '%d %b, %Y', '%Y-%m-%d', '%B %d, %Y']:
        try:
            return pd.Timestamp(datetime.strptime(date_str, fmt))
        except:
            continue
    return pd.NaT

df['release_date_parsed'] = df['release_date'].apply(parse_release_date)
df['release_date_parsed'] = pd.to_datetime(df['release_date_parsed'], utc=True)

now_utc = pd.Timestamp.now(tz='UTC')
df['days_since_release'] = (now_utc - df['release_date_parsed']).dt.days
df['days_since_release'] = df['days_since_release'].fillna(-1).astype(int)
df['release_year'] = df['release_date_parsed'].dt.year
df['release_month'] = df['release_date_parsed'].dt.month
df['is_legacy'] = (df['days_since_release'] > 1825).astype(int)
print(f"✓ Age features: {(df['days_since_release'] >= 0).sum():,} valid dates")

✓ Age features: 4,924 valid dates


In [54]:
# Price features -- ITAD-only (Option A).
#
# Source of truth: price_history (collected with country='US', so prices are
# in USD, history goes back to 2012-2015). The Steam Storefront columns
# launch_price_cents and current_price_cents on the games table are no
# longer used here -- we derive both initial and current price from ITAD's
# latest snapshot per game. This eliminates the previous mixed-currency
# arithmetic that produced the broken `true_discount_depth` column.

df_price_history = pd.read_sql_query("SELECT * FROM price_history", conn)
df_price_history['timestamp'] = pd.to_datetime(df_price_history['timestamp'], utc=True)

# 1. Latest snapshot per game -> current state
latest_snapshot = (
    df_price_history.sort_values('timestamp')
    .groupby('appid').last()
    .reset_index()
    [['appid', 'price_amount', 'regular_amount', 'cut', 'price_currency']]
    .rename(columns={
        'price_amount':  'current_price',     # USD: what the game costs right now
        'regular_amount': 'initial_price',    # USD: regular MSRP (essentially launch price)
        'cut':           'current_cut',
    })
)

# 2. Aggregate stats over full history
price_stats = df_price_history.groupby('appid').agg(
    min_historical_price = ('price_amount', 'min'),
    max_historical_price = ('price_amount', 'max'),
    avg_historical_price = ('price_amount', 'mean'),
    std_historical_price = ('price_amount', 'std'),
    max_discount_ever    = ('cut',          'max'),
    avg_discount         = ('cut',          'mean'),
    price_change_count   = ('timestamp',    'count'),
).reset_index()

df = df.merge(latest_snapshot, on='appid', how='left')
df = df.merge(price_stats,     on='appid', how='left')
df['has_dynamic_pricing'] = df['min_historical_price'].notna().astype(int)

# 3. Discount depth from current snapshot's cut% (currency-free, exact)
#    Equivalent to (initial_price - current_price) / initial_price; using
#    cut directly avoids float drift.
df['discount_depth'] = (df['current_cut'] / 100).fillna(0).clip(0, 1)

# 4. ever_discounted: 1 if any cut > 0 was ever observed for this game
ever_disc = (
    df_price_history[df_price_history['cut'] > 0]
    .groupby('appid').size().gt(0).astype(int).rename('ever_discounted')
)
df = df.merge(ever_disc, on='appid', how='left')
df['ever_discounted'] = df['ever_discounted'].fillna(0).astype(int)

# 5. Price tier in USD
def categorize_price_tier(price):
    if pd.isna(price) or price == 0:
        return 'Free'
    elif price < 5:
        return 'Budget'      # under $5
    elif price < 30:
        return 'Mid'         # $5 - $30
    else:
        return 'Premium'     # $30 and up

df['price_tier'] = df['initial_price'].apply(categorize_price_tier)

# 6. Discount frequency bucket
def categorize_discount_freq(count):
    if pd.isna(count) or count == 0:
        return 'Never'
    elif count < 3:
        return 'Rare'
    elif count < 6:
        return 'Occasional'
    else:
        return 'Frequent'

df['discount_frequency'] = df['price_change_count'].apply(categorize_discount_freq)
print(f"[OK] Price features (ITAD-only, USD): {df['has_dynamic_pricing'].sum():,} of {len(df):,} games have ITAD coverage")

[OK] Price features (ITAD-only, USD): 4,904 of 4,946 games have ITAD coverage


In [55]:
# Reception features
df['review_score'] = 0.0
mask = (df['total_positive'] + df['total_negative']) > 0
df.loc[mask, 'review_score'] = df.loc[mask, 'total_positive'] / (df.loc[mask, 'total_positive'] + df.loc[mask, 'total_negative'])

df['ownership_midpoint'] = (df['owners_min'] + df['owners_max']) / 2
df['owners'] = df['ownership_midpoint']  # Alias
df['log_ownership'] = np.log10(df['ownership_midpoint'] + 1)
print("✓ Reception features created")

✓ Reception features created


In [56]:
# Categorical features
df['primary_genre'] = df['genres'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else 'Unknown')

# IS_MULTIPLAYER
def check_multiplayer(categories):
    if not isinstance(categories, list):
        return 0
    multiplayer_keywords = ['Multi-player', 'Multiplayer', 'Online PvP', 'Online Co-op', 'Co-op']
    return int(any(keyword in str(cat) for cat in categories for keyword in multiplayer_keywords))

df['is_multiplayer'] = df['categories'].apply(check_multiplayer)

# HAS_CONTROLLER_SUPPORT
df['has_controller_support'] = df['controller_support'].notna().astype(int)

# Developer tier
AAA = ['Valve', 'EA', 'Ubisoft', 'Activision', 'Rockstar', 'Bethesda', 'Square Enix', 'Capcom']
df['developer_tier'] = df.apply(lambda r: 'AAA' if any(a in str(r['developer']) for a in AAA) else ('Indie' if r['ownership_midpoint'] < 50000 else 'Mid-tier'), axis=1)

print(f"✓ Categorical features created")
print(f"  - is_multiplayer: {df['is_multiplayer'].sum():,} games")
print(f"  - has_controller_support: {df['has_controller_support'].sum():,} games")

✓ Categorical features created
  - is_multiplayer: 2,260 games
  - has_controller_support: 1,950 games


In [57]:
# Boxleiter
multiplier_map = {'AAA': 30, 'Mid-tier': 65, 'Indie': 40}
df['boxleiter_multiplier'] = df['developer_tier'].map(multiplier_map).fillna(40)
df['units_sold_estimate'] = (df['total_reviews'] * df['boxleiter_multiplier']).fillna(0).astype(int)
print("✓ Boxleiter calculated")

✓ Boxleiter calculated


In [58]:
# Player features
print("Loading ALL player data...")
df_players = pd.read_sql_query("SELECT * FROM steamcharts_history", conn)
print(f"  Loaded {len(df_players):,} records for {df_players['appid'].nunique():,} games")

if len(df_players) > 0:
    df_players['timestamp'] = pd.to_datetime(df_players['timestamp'], utc=True)
    recent = df_players[df_players['timestamp'] >= df_players['timestamp'].max() - timedelta(days=30)]
    stats = recent.groupby('appid')['player_count'].agg(['mean', 'max']).reset_index()
    stats.columns = ['appid', 'avg_recent_players', 'max_recent_players']
    df = df.merge(stats, on='appid', how='left')
    
    def categorize_player_engagement(avg):
        if pd.isna(avg):
            return 'Unknown'
        elif avg < 100:
            return 'Low'
        elif avg < 1000:
            return 'Medium'
        else:
            return 'High'
    
    df['player_engagement'] = df['avg_recent_players'].apply(categorize_player_engagement)
else:
    df['avg_recent_players'] = np.nan
    df['player_engagement'] = 'Unknown'
print(f"✓ Player features: {df['avg_recent_players'].notna().sum():,} games")

Loading ALL player data...
  Loaded 3,085,925 records for 4,924 games
✓ Player features: 4,922 games


In [59]:
# Target: value retention
df['price_retention_ratio'] = 0.0
mask = (df['initial_price'] > 0) & (df['days_since_release'] >= 365)
df.loc[mask, 'price_retention_ratio'] = df.loc[mask, 'current_price'] / df.loc[mask, 'initial_price']
df['price_retention'] = df['price_retention_ratio']  # Alias

def categorize_value_retention(ratio, age):
    if age < 365:
        return 'Too New'
    elif ratio > 0.85:
        return 'Premium Hold'
    elif ratio >= 0.50:
        return 'Standard Depreciation'
    elif ratio >= 0.25:
        return 'Heavy Discount'
    else:
        return 'Permanent Bargain'

df['value_retention_tier'] = df.apply(lambda r: categorize_value_retention(r['price_retention_ratio'], r['days_since_release']), axis=1)
print("✓ Target created")

✓ Target created


In [60]:
# Save Part 1
cols_part1 = [
    'appid', 'title', 'developer', 'publisher', 'release_date',
    'days_since_release', 'release_year', 'release_month',
    'initial_price', 'current_price', 'discount_depth', 'price_tier',
    'price_retention_ratio', 'price_retention', 'value_retention_tier',
    'review_score', 'total_reviews', 'total_positive', 'total_negative',
    'ownership_midpoint', 'owners', 'log_ownership', 'units_sold_estimate',
    'primary_genre', 'developer_tier', 'is_multiplayer', 'has_controller_support',
    'achievements_total', 'ever_discounted', 'max_discount_ever',
    'player_engagement', 'avg_recent_players'
]

df_part1 = df[cols_part1].copy()
df_part1.to_csv('part1_game_level_cleaned.csv', index=False)
print(f"\n✅ PART 1 SAVED: {len(df_part1):,} games, {len(df_part1.columns)} columns")


✅ PART 1 SAVED: 4,946 games, 32 columns


---
# PART 2: SALE-EVENT LEVEL
---

In [61]:
print("\nCREATING SALE EVENTS...")
df_price_history['timestamp'] = pd.to_datetime(df_price_history['timestamp'], utc=True)
df_sales = df_price_history[df_price_history['cut'] > 0].sort_values(['appid', 'timestamp'])
print(f"Sale records: {len(df_sales):,}")


CREATING SALE EVENTS...


Sale records: 224,734


In [62]:
# Group consecutive sales
events = []
for appid in df_sales['appid'].unique():
    game = df_sales[df_sales['appid'] == appid].sort_values('timestamp')
    if len(game) == 0: continue
    
    curr = {'appid': appid, 'start_date': game.iloc[0]['timestamp'], 'end_date': game.iloc[0]['timestamp'], 'max_discount': game.iloc[0]['cut'], 'sale_price': game.iloc[0]['price_amount'], 'regular_price': game.iloc[0]['regular_amount']}
    
    for i in range(1, len(game)):
        row = game.iloc[i]
        if (row['timestamp'] - curr['end_date']).days <= 7:
            curr['end_date'] = row['timestamp']
            curr['max_discount'] = max(curr['max_discount'], row['cut'])
        else:
            events.append(curr)
            curr = {'appid': appid, 'start_date': row['timestamp'], 'end_date': row['timestamp'], 'max_discount': row['cut'], 'sale_price': row['price_amount'], 'regular_price': row['regular_amount']}
    events.append(curr)

df_events = pd.DataFrame(events)
print(f"✓ Created {len(df_events):,} sale events")

✓ Created 213,959 sale events


In [63]:
# Event features
df_events['sale_duration_days'] = (df_events['end_date'] - df_events['start_date']).dt.days + 1
df_events['discount_depth'] = df_events['max_discount'] / 100
df_events['discount_percent'] = df_events['max_discount']

# Prices already in PHP from ITAD (we passed country=PH at collection time)
df_events['sale_price_php'] = df_events['sale_price']
df_events['regular_price_php'] = df_events['regular_price']

df_events = df_events.merge(df[['appid', 'release_date_parsed']], on='appid', how='left')
df_events['age_at_sale_days'] = (df_events['start_date'] - df_events['release_date_parsed']).dt.days

# Days since last sale & cumulative count
df_events = df_events.sort_values(['appid', 'start_date'])
df_events['days_since_last_sale'] = -1
df_events['cumulative_sale_count'] = 0

for appid in df_events['appid'].unique():
    mask = df_events['appid'] == appid
    game = df_events[mask].sort_values('start_date')
    for i in range(1, len(game)):
        df_events.loc[game.index[i], 'days_since_last_sale'] = (game.iloc[i]['start_date'] - game.iloc[i-1]['end_date']).days
    df_events.loc[mask, 'cumulative_sale_count'] = range(len(game))

# Sale type
df_events['sale_type'] = df_events.apply(lambda r: ('Summer Sale' if r['start_date'].month in [6,7] else ('Winter Sale' if r['start_date'].month in [11,12] else ('Weekend Deal' if r['sale_duration_days'] <= 4 else 'Publisher Sale'))), axis=1)
print("✓ Event features created")

✓ Event features created


In [64]:
# Player uplift -- TIGHTER WINDOWS to reduce decay bias.
#
# Old windows had baseline 30-60d before sale and during from 14d before to
# 14d after sale_end. Because the during window sat 30-60d further into each
# game's natural decay curve than the baseline, most events appeared to have
# negative uplift even when sales did boost players. Fix: shrink the gap
# between the two windows so natural decay can't dominate.
#
#   baseline = [sale_start - 30d, sale_start -  7d]   (23-day window, ends 7d before sale)
#   during   = [sale_start,        sale_end   + 7d]   (sale window + 7d cooldown)
#
# 7-day buffers prevent pre-sale leak from contaminating the baseline and
# post-sale carry-over from contaminating the during window.

print("\nCALCULATING UPLIFT (tightened windows)...")
print(f"Events: {len(df_events):,}")
print(f"Player records: {len(df_players):,}")
print("Estimated time: 15-20 minutes\n")

uplifts = []
successful = 0

for idx, sale in df_events.iterrows():
    if idx % 500 == 0:
        print(f"  {idx:,}/{len(df_events):,} ({idx/len(df_events)*100:.1f}%) - Success: {successful:,}")

    try:
        game_players = df_players[df_players['appid'] == sale['appid']]
        if len(game_players) == 0:
            uplifts.append({'player_count_baseline': np.nan, 'player_count_during_sale': np.nan, 'uplift_percent': np.nan})
            continue

        baseline = game_players[
            (game_players['timestamp'] >= sale['start_date'] - timedelta(days=30)) &
            (game_players['timestamp'] <= sale['start_date'] - timedelta(days=7))
        ]
        during = game_players[
            (game_players['timestamp'] >= sale['start_date']) &
            (game_players['timestamp'] <= sale['end_date'] + timedelta(days=7))
        ]

        if len(baseline) == 0 or len(during) == 0:
            uplifts.append({'player_count_baseline': np.nan, 'player_count_during_sale': np.nan, 'uplift_percent': np.nan})
            continue

        b_avg, d_avg = baseline['player_count'].mean(), during['player_count'].mean()
        uplift = ((d_avg - b_avg) / b_avg * 100) if b_avg > 0 else np.nan
        uplifts.append({'player_count_baseline': b_avg, 'player_count_during_sale': d_avg, 'uplift_percent': uplift})
        successful += 1

    except Exception:
        uplifts.append({'player_count_baseline': np.nan, 'player_count_during_sale': np.nan, 'uplift_percent': np.nan})

print(f"  {len(df_events):,}/{len(df_events):,} (100.0%) - Success: {successful:,}")

df_events = pd.concat([df_events.reset_index(drop=True), pd.DataFrame(uplifts)], axis=1)
print(f"\nUplift coverage: {successful:,}/{len(df_events):,} ({successful/len(df_events)*100:.1f}%)")

# Distribution check -- should now be more balanced around 0 with a rightward
# tail of genuine sale spikes, instead of skewing heavily negative.
valid = df_events['uplift_percent'].dropna()
print(f"Uplift distribution: median {valid.median():.1f}%,  mean {valid.mean():.1f}%,  "
      f"% positive {(valid > 0).mean()*100:.1f}%")


CALCULATING UPLIFT (tightened windows)...
Events: 213,959
Player records: 3,085,925
Estimated time: 15-20 minutes

  0/213,959 (0.0%) - Success: 0
  500/213,959 (0.2%) - Success: 171
  1,000/213,959 (0.5%) - Success: 361
  1,500/213,959 (0.7%) - Success: 539
  2,000/213,959 (0.9%) - Success: 694
  2,500/213,959 (1.2%) - Success: 849
  3,000/213,959 (1.4%) - Success: 1,013
  3,500/213,959 (1.6%) - Success: 1,147
  4,000/213,959 (1.9%) - Success: 1,323
  4,500/213,959 (2.1%) - Success: 1,462
  5,000/213,959 (2.3%) - Success: 1,633
  5,500/213,959 (2.6%) - Success: 1,801
  6,000/213,959 (2.8%) - Success: 1,974
  6,500/213,959 (3.0%) - Success: 2,146
  7,000/213,959 (3.3%) - Success: 2,300
  7,500/213,959 (3.5%) - Success: 2,443
  8,000/213,959 (3.7%) - Success: 2,604
  8,500/213,959 (4.0%) - Success: 2,748
  9,000/213,959 (4.2%) - Success: 2,886
  9,500/213,959 (4.4%) - Success: 3,026
  10,000/213,959 (4.7%) - Success: 3,173
  10,500/213,959 (4.9%) - Success: 3,349
  11,000/213,959 (5.1%

In [65]:
# Target
def categorize_sale_effectiveness(uplift):
    if pd.isna(uplift):
        return 'Unknown'
    elif uplift < 5:
        return 'None'
    elif uplift < 20:
        return 'Low'
    elif uplift < 50:
        return 'Moderate'
    else:
        return 'High Impact'

df_events['sale_effectiveness_tier'] = df_events['uplift_percent'].apply(categorize_sale_effectiveness)
df_events['uplift_tier'] = df_events['sale_effectiveness_tier']  # Alias for notebooks
print("✓ Target created")

✓ Target created


In [66]:
# Merge ALL game features
game_feats = [
    'appid', 'title', 'developer',
    'primary_genre', 'developer_tier', 'price_tier',
    'review_score', 'total_reviews',
    'ownership_midpoint', 'owners', 'log_ownership',
    'is_multiplayer', 'has_controller_support',
    'player_engagement', 'avg_recent_players',
    'days_since_release', 'release_year'
]

df_events_final = df_events.merge(df[game_feats], on='appid', how='left')

# Aliases for notebook compatibility
df_events_final['baseline_players'] = df_events_final['player_count_baseline']
df_events_final['during_players'] = df_events_final['player_count_during_sale']

print(f"✓ Merged: {len(df_events_final):,} events with all features")

✓ Merged: 213,959 events with all features


In [67]:
# Save Part 2
cols_part2 = [
    'appid', 'title', 'developer',
    'start_date', 'end_date', 'sale_duration_days',
    'discount_depth', 'discount_percent', 'sale_type',
    'age_at_sale_days', 'days_since_last_sale', 'cumulative_sale_count',
    'days_since_release', 'release_year',
    'primary_genre', 'developer_tier', 'price_tier',
    'review_score', 'total_reviews',
    'ownership_midpoint', 'owners', 'log_ownership',
    'is_multiplayer', 'has_controller_support',
    'player_engagement', 'avg_recent_players',
    'player_count_baseline', 'baseline_players',
    'player_count_during_sale', 'during_players',
    'uplift_percent', 'uplift_tier', 'sale_effectiveness_tier'
]

df_part2 = df_events_final[cols_part2].copy()
df_part2.to_csv('part2_sale_event_level_cleaned.csv', index=False)
print(f"\n✅ PART 2 SAVED: {len(df_part2):,} events, {len(df_part2.columns)} columns")


✅ PART 2 SAVED: 213,959 events, 33 columns


---
# PART 3: AGE-PANEL DISCOUNT TRAJECTORY
---

For each game, compute peak discount reached in each year-of-life bucket. Output is panel-shaped: one row per `(appid, age_year)` pair. This becomes the input for a regression that predicts the depreciation curve of a new game given its attributes.

**Why panel over a single summary stat:** discount depth grows with age, but the slope varies by genre, dev tier, and reception. A static median collapses that trajectory; a panel preserves it so the model can learn discount-vs-age conditional on game attributes.

In [68]:
# Build the (appid, age_year) panel from price_history.
print('Building age-panel...')

# 1. Annotate every price snapshot with the game's age (in years) at observation
ph = df_price_history.merge(
    df[['appid', 'release_date_parsed']],
    on='appid',
    how='left',
)
ph['observation_age_days']  = (ph['timestamp'] - ph['release_date_parsed']).dt.days
ph = ph[ph['observation_age_days'] >= 0]   # drop pre-release snapshots if any
ph['age_year']              = (ph['observation_age_days'] // 365).astype(int)

# Cap at 10+ years - earlier years have most signal, beyond ~10 the data thins out
ph['age_year'] = ph['age_year'].clip(upper=10)

# 2. Aggregate per (appid, age_year)
panel = ph.groupby(['appid', 'age_year']).agg(
    max_discount      = ('cut',          'max'),
    mean_discount     = ('cut',          'mean'),
    n_observations    = ('cut',          'count'),
    median_sale_price = ('price_amount', 'median'),
).reset_index()

# Drop sparse buckets - with <2 snapshots the max_discount is unreliable
# Keep only buckets where the game was actually observed on sale that year.
# (Old filter required n_observations >= 2 — but ITAD coverage is sparse, so
# most games had only 1 snapshot per year-bucket and got dropped. Result: median
# rows per game = 1, which collapses the panel into a non-panel. Filtering on
# max_discount > 0 keeps every year where a sale was genuinely observed,
# preserving the trajectory for ITAD-rich games.)
panel = panel[panel['max_discount'] > 0].copy()

print(f'  Raw panel rows:            {len(panel):,}')
print(f'  Unique games in panel:     {panel["appid"].nunique():,}')
print(f'  Median rows per game:      {panel.groupby("appid").size().median():.0f}')
print(f'  Age range:                 {panel["age_year"].min()} - {panel["age_year"].max()} years')

Building age-panel...
  Raw panel rows:            27,782
  Unique games in panel:     4,083
  Median rows per game:      7
  Age range:                 0 - 10 years


In [69]:
# Merge game attributes onto every panel row (time-invariant covariates).
# review_score and ownership are technically time-varying but we use snapshot
# values as a simplification - a stronger version would track review velocity.
panel_features = panel.merge(
    df[[
        'appid', 'title',
        'initial_price',
        'primary_genre', 'developer_tier', 'price_tier',
        'review_score', 'log_ownership',
        'is_multiplayer', 'has_controller_support', 'achievements_total',
    ]],
    on='appid',
    how='left',
)

# Buyer-value ratio at this age: price you'd pay during a typical sale this year
# divided by launch price. NaN when no sales observed in the bucket.
panel_features['buyer_value_at_age'] = (
    panel_features['median_sale_price'] / panel_features['initial_price']
).clip(0, 1)

print(f'Panel with features: {len(panel_features):,} rows x {len(panel_features.columns)} columns')
panel_features.head(8)

Panel with features: 27,782 rows x 17 columns


,appid,age_year,max_discount,mean_discount,n_observations,median_sale_price,title,initial_price,primary_genre,developer_tier,price_tier,review_score,log_ownership,is_multiplayer,has_controller_support,achievements_total,buyer_value_at_age
0,10,10,90,41.712329,73,4.990,Counter-Strike,9.99,Action,AAA,Mid,0.974155,7.176091,1,0,0,0.499499
1,20,10,90,37.951923,104,4.365,Team Fortress Classic,4.99,Action,AAA,Budget,0.871120,6.176092,1,0,0,0.874749
2,30,9,75,62.500000,4,3.115,Day of Defeat,4.99,Action,AAA,Budget,0.902108,6.875061,1,0,0,0.624248
3,30,10,90,38.500000,100,3.865,Day of Defeat,4.99,Action,AAA,Budget,0.902108,6.875061,1,0,0,0.774549
4,40,10,90,39.088235,102,3.740,Deathmatch Classic,4.99,Action,AAA,Budget,0.832061,6.875061,1,0,0,0.749499
5,50,10,90,39.018182,110,3.740,Half-Life: Opposing Force,4.99,Action,Mid-tier,Budget,0.953408,6.544068,1,0,0,0.749499
6,60,10,90,39.201923,104,3.740,Ricochet,4.99,Action,AAA,Budget,0.830891,5.544069,1,0,0,0.749499
7,70,10,100,41.528926,121,4.990,Half-Life,9.99,Action,AAA,Mid,0.965072,7.176091,1,1,0,0.499499


In [70]:
# Save Part 3 — panel data
panel_features.to_csv('part3_age_panel_cleaned.csv', index=False)
print(f"\n✅ PART 3 SAVED: {len(panel_features):,} (game, age_year) rows, {len(panel_features.columns)} columns")


✅ PART 3 SAVED: 27,782 (game, age_year) rows, 17 columns


In [71]:
print("\nSaving to database...")
df_part1.to_sql('cleaned_games',         conn, if_exists='replace', index=False)
df_part2.to_sql('cleaned_sale_events',   conn, if_exists='replace', index=False)
panel_features.to_sql('cleaned_discount_panel', conn, if_exists='replace', index=False)
print(f"✓ Database updated: cleaned_games, cleaned_sale_events, cleaned_discount_panel")


Saving to database...
✓ Database updated: cleaned_games, cleaned_sale_events, cleaned_discount_panel


In [72]:
conn.close()
print("\n" + "="*80)
print("🎉 PHASE 2 COMPLETE!")
print("="*80)
print(f"\nPart 1: {len(df_part1):,} games          → For EDA & ML Part 1 (current state)")
print(f"Part 2: {len(df_part2):,} sale events    → For ML Part 2 (sale effectiveness)")
print(f"Part 3: {len(panel_features):,} (game, age_year) panel rows  → For ML Part 1 panel regression (depreciation curve)")
print(f"\nUplift coverage: {successful:,}/{len(df_part2):,} ({successful/len(df_part2)*100:.1f}%)")
print(f"\n✅ Ready to run notebooks!")


🎉 PHASE 2 COMPLETE!

Part 1: 4,946 games          → For EDA & ML Part 1 (current state)
Part 2: 213,959 sale events    → For ML Part 2 (sale effectiveness)
Part 3: 27,782 (game, age_year) panel rows  → For ML Part 1 panel regression (depreciation curve)

Uplift coverage: 71,080/213,959 (33.2%)

✅ Ready to run notebooks!
